In [1]:
import pandas as pd

In [55]:
df1 = pd.read_csv("/Users/dana/Desktop/hydrogen_storage/data/1/weight_percent.csv")
df1.columns

Index(['Composition', 'hydrogen_weight_percent', 'heat_of_formation',
       'temperature', 'pressure', 'avg_atomic_mass'],
      dtype='object')

In [56]:
df2 = pd.read_csv("/Users/dana/Desktop/hydrogen_storage/data/2/ML-HYDPARK_v0.0.5.csv")
df2.columns

Index(['Unnamed: 0', 'index', 'Material_Class', 'Composition_Formula',
       'Hydrogen_Weight_Percent', 'Heat_of_Formation_kJperMolH2',
       'Temperature_oC', 'Pressure_Atmospheres_Absolute',
       'Entropy_of_Formation_JperMolH2perK', 'Equilibrium_Pressure_25C',
       'LnEquilibrium_Pressure_25C', 'HtoM', 'Reference'],
      dtype='object')

In [57]:
df3 = pd.read_csv("/Users/dana/Desktop/hydrogen_storage/data/3/table3.csv")
df3.columns

Index(['NO', 'Ti', 'Fe', 'Zr', 'V', 'Cr', 'Mn', 'Ni', 'Co', 'Y', 'T', 'Size1',
       'Size2', 'Pa', 'Pd', 'wt', 'ref'],
      dtype='object')

## First, let's write the compostiion as columns for df1, df2 as df3

In [5]:
pip install pymatgen pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.5.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.68.4-py3-none-any.whl.metadata (57 kB)
  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 4.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 12.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 27.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 27.1 MB/s  0:00:00 eta 0:00:01
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 911.1/911.1 kB 21.7 MB/s  0:00:00
Using cached tqdm-4.68.4-py3-none-any.whl (676 kB)
Using cached ruamel_yaml-0.19.1-py3-none-any.whl (118 kB)
  Created wheel for bibtexp

In [58]:
from pymatgen.core import Composition

def formula_to_columns(formula):
    return dict(Composition(formula).get_el_amt_dict())

# Parse each formula into a dict of {element: amount}, expand into columns
elements = df1["Composition"].apply(formula_to_columns).apply(pd.Series)
elements = elements.fillna(0)

df = pd.concat([df1, elements], axis=1)
df.to_csv("your_file_parsed.csv", index=False)
df.head()

,Composition,hydrogen_weight_percent,heat_of_formation,temperature,pressure,avg_atomic_mass,Ti,V,Nb,Cr,...,N,C,Au,Ga,In,Os,Re,Sn,Ge,La
0,(TiVNb)65Cr35,2.80,NaN,24.0,NaN,62.092742,65.0,65.0,65.0,35.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,(TiVNb)75Cr25,2.70,NaN,24.0,NaN,62.714074,75.0,75.0,75.0,25.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,(TiVNb)85Cr15,3.18,NaN,25.0,NaN,63.243357,85.0,85.0,85.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,(TiVNb)85Cr15,3.00,NaN,24.0,NaN,63.243357,85.0,85.0,85.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,(TiVNb)85Cr15,3.18,NaN,20.0,NaN,63.243357,85.0,85.0,85.0,15.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
import re
from pymatgen.core import Composition

def parse_alloy(formula):
    # fix leading-dot decimals: "V.89" -> "V0.89", ").95" -> ")0.95"
    formula = re.sub(r"(?<!\d)\.(\d)", r"0.\1", formula)

    total = {}
    bracket = r"\(([^)]+)\)(\d*\.?\d*)"

    # bracket groups: normalize inside to 1, then multiply by outer coefficient
    for inner, coeff in re.findall(bracket, formula):
        coeff = float(coeff) if coeff else 1.0
        for el, amt in Composition(inner).fractional_composition.get_el_amt_dict().items():
            total[el] = total.get(el, 0) + amt * coeff

    # whatever is left outside brackets (e.g. "Cr35")
    rest = re.sub(bracket, "", formula).strip()
    if rest:
        for el, amt in Composition(rest).get_el_amt_dict().items():
            total[el] = total.get(el, 0) + amt

    # normalize whole composition to sum = 1
    s = sum(total.values())
    return {el: round(amt / s, 6) for el, amt in total.items()}

In [60]:
df.columns

Index(['Composition', 'hydrogen_weight_percent', 'heat_of_formation',
       'temperature', 'pressure', 'avg_atomic_mass', 'Ti', 'V', 'Nb', 'Cr',
       'Co', 'Ni', 'Fe', 'Al', 'Mn', 'Zr', 'Cu', 'Ag', 'Pd', 'W', 'Mg', 'Zn',
       'Bi', 'Hf', 'Li', 'B', 'Mo', 'Si', 'Rh', 'Pt', 'Ir', 'Y', 'Ce', 'Pb',
       'Sb', 'Sc', 'U', 'Ru', 'Ta', 'N', 'C', 'Au', 'Ga', 'In', 'Os', 'Re',
       'Sn', 'Ge', 'La'],
      dtype='object')

In [61]:
import pandas as pd

ELEMENTS = ['Ti', 'V', 'Nb', 'Cr', 'Co', 'Ni', 'Fe', 'Al', 'Mn', 'Zr', 'Cu',
            'Ag', 'Pd', 'W', 'Mg', 'Zn', 'Bi', 'Hf', 'Li', 'B', 'Mo', 'Si',
            'Rh', 'Pt', 'Ir', 'Y', 'Ce', 'Pb', 'Sb', 'Sc', 'U', 'Ru', 'Ta',
            'N', 'C', 'Au', 'Ga', 'In', 'Os', 'Re', 'Sn', 'Ge', 'La']

elements = df["Composition"].apply(parse_alloy).apply(pd.Series)
elements = elements.reindex(columns=ELEMENTS, fill_value=0).fillna(0)
df = pd.concat([df, elements], axis=1)
df = df.loc[:, ~df.columns.duplicated(keep="last")]

df.head()

,Composition,hydrogen_weight_percent,heat_of_formation,temperature,pressure,avg_atomic_mass,Ti,V,Nb,Cr,...,N,C,Au,Ga,In,Os,Re,Sn,Ge,La
0,(TiVNb)65Cr35,2.80,NaN,24.0,NaN,62.092742,0.216667,0.216667,0.216667,0.35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,(TiVNb)75Cr25,2.70,NaN,24.0,NaN,62.714074,0.250000,0.250000,0.250000,0.25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,(TiVNb)85Cr15,3.18,NaN,25.0,NaN,63.243357,0.283333,0.283333,0.283333,0.15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,(TiVNb)85Cr15,3.00,NaN,24.0,NaN,63.243357,0.283333,0.283333,0.283333,0.15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,(TiVNb)85Cr15,3.18,NaN,20.0,NaN,63.243357,0.283333,0.283333,0.283333,0.15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [62]:
parsed2 = df2["Composition_Formula"].apply(parse_alloy).apply(pd.Series)

print(sorted(parsed2.columns))                      # all elements in df2

['Ag', 'Al', 'B', 'Be', 'Bi', 'Ca', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu', 'Fe', 'Ga', 'Gd', 'Ge', 'Hf', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mg', 'Mn', 'Mo', 'Nb', 'Nd', 'Ni', 'O', 'Pd', 'Pr', 'Pt', 'Rh', 'Ru', 'Sb', 'Sc', 'Si', 'Sm', 'Sn', 'Ta', 'Tb', 'Th', 'Ti', 'Tm', 'U', 'V', 'W', 'Y', 'Zn', 'Zr']


In [63]:
ELEMENTS = sorted(set(ELEMENTS) | set(parsed2.columns))  

# df2
elements2 = parsed2.reindex(columns=ELEMENTS, fill_value=0).fillna(0)
df2 = pd.concat([df2, elements2], axis=1)
df2.columns

Index(['Unnamed: 0', 'index', 'Material_Class', 'Composition_Formula',
       'Hydrogen_Weight_Percent', 'Heat_of_Formation_kJperMolH2',
       'Temperature_oC', 'Pressure_Atmospheres_Absolute',
       'Entropy_of_Formation_JperMolH2perK', 'Equilibrium_Pressure_25C',
       'LnEquilibrium_Pressure_25C', 'HtoM', 'Reference', 'Ag', 'Al', 'Au',
       'B', 'Be', 'Bi', 'C', 'Ca', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu',
       'Fe', 'Ga', 'Gd', 'Ge', 'Hf', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mg',
       'Mn', 'Mo', 'N', 'Nb', 'Nd', 'Ni', 'O', 'Os', 'Pb', 'Pd', 'Pr', 'Pt',
       'Re', 'Rh', 'Ru', 'Sb', 'Sc', 'Si', 'Sm', 'Sn', 'Ta', 'Tb', 'Th', 'Ti',
       'Tm', 'U', 'V', 'W', 'Y', 'Zn', 'Zr'],
      dtype='object')

In [64]:
df2 = df2.loc[:, ~df2.columns.duplicated(keep="last")]

In [65]:
df2.head()

,Unnamed: 0,index,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK,Equilibrium_Pressure_25C,...,Tb,Th,Ti,Tm,U,V,W,Y,Zn,Zr
0,0,0,A2B,Th2Al,0.8,130.0,500.0,0.001,110.711134,1.016736e-17,...,0.0,0.666667,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,1,1,A2B,Ti2Cu,2.2,130.0,500.0,0.120,150.515102,1.220083e-15,...,0.0,0.000000,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,2,2,A2B,Zr2Cu,1.3,144.0,600.0,0.003,116.621978,7.297612e-20,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.666667
3,3,3,A2B,Zr2Ni,1.3,183.0,604.0,0.003,160.332084,2.058957e-24,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.666667
4,4,4,A2B,Mg2Ni,3.6,64.5,299.0,3.200,122.403296,1.240158e-05,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [66]:
print(df.columns.tolist())
print(df2.columns.tolist())
print(df3.columns.tolist())

['Composition', 'hydrogen_weight_percent', 'heat_of_formation', 'temperature', 'pressure', 'avg_atomic_mass', 'Ti', 'V', 'Nb', 'Cr', 'Co', 'Ni', 'Fe', 'Al', 'Mn', 'Zr', 'Cu', 'Ag', 'Pd', 'W', 'Mg', 'Zn', 'Bi', 'Hf', 'Li', 'B', 'Mo', 'Si', 'Rh', 'Pt', 'Ir', 'Y', 'Ce', 'Pb', 'Sb', 'Sc', 'U', 'Ru', 'Ta', 'N', 'C', 'Au', 'Ga', 'In', 'Os', 'Re', 'Sn', 'Ge', 'La']
['Unnamed: 0', 'index', 'Material_Class', 'Composition_Formula', 'Hydrogen_Weight_Percent', 'Heat_of_Formation_kJperMolH2', 'Temperature_oC', 'Pressure_Atmospheres_Absolute', 'Entropy_of_Formation_JperMolH2perK', 'Equilibrium_Pressure_25C', 'LnEquilibrium_Pressure_25C', 'HtoM', 'Reference', 'Ag', 'Al', 'Au', 'B', 'Be', 'Bi', 'C', 'Ca', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu', 'Fe', 'Ga', 'Gd', 'Ge', 'Hf', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mg', 'Mn', 'Mo', 'N', 'Nb', 'Nd', 'Ni', 'O', 'Os', 'Pb', 'Pd', 'Pr', 'Pt', 'Re', 'Rh', 'Ru', 'Sb', 'Sc', 'Si', 'Sm', 'Sn', 'Ta', 'Tb', 'Th', 'Ti', 'Tm', 'U', 'V', 'W', 'Y', 'Zn', 'Zr']
['NO', 

In [67]:
all_elements = ['Ti', 'Fe', 'Zr', 'V', 'Cr', 'Mn', 'Ni', 'Co', 'Y', 'T','Ag', 'Al', 'Au', 'B', 'Be', 'Bi', 'C', 'Ca', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu', 'Fe', 'Ga', 'Gd', 'Ge', 'Hf', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mg', 'Mn', 'Mo', 'N', 'Nb', 'Nd', 'Ni', 'O', 'Os', 'Pb', 'Pd', 'Pr', 'Pt', 'Re', 'Rh', 'Ru', 'Sb', 'Sc', 'Si', 'Sm', 'Sn', 'Ta', 'Tb', 'Th', 'Ti', 'Tm', 'U', 'V', 'W', 'Y', 'Zn', 'Zr', 'Ag', 'Al', 'Au', 'B', 'Be', 'Bi', 'C', 'Ca', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu', 'Fe', 'Ga', 'Gd', 'Ge', 'Hf', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mg', 'Mn', 'Mo', 'N', 'Nb', 'Nd', 'Ni', 'O', 'Os', 'Pb', 'Pd', 'Pr', 'Pt', 'Re', 'Rh', 'Ru', 'Sb', 'Sc', 'Si', 'Sm', 'Sn', 'Ta', 'Tb', 'Th', 'Ti', 'Tm', 'U', 'V', 'W', 'Y', 'Zn', 'Zr', 'Ti', 'V', 'Nb', 'Cr', 'Co', 'Ni', 'Fe', 'Al', 'Mn', 'Zr', 'Cu', 'Ag', 'Pd', 'W', 'Mg', 'Zn', 'Bi', 'Hf', 'Li', 'B', 'Mo', 'Si', 'Rh', 'Pt', 'Ir', 'Y', 'Ce', 'Pb', 'Sb', 'Sc', 'U', 'Ru', 'Ta', 'N', 'C', 'Au', 'Ga', 'In', 'Os', 'Re', 'Sn', 'Ge', 'La' ]
all_elements = sorted(set(all_elements))

In [68]:
all_elements = sorted(set(all_elements) - {"T"})

In [69]:
element_cols3 = ['Ti', 'Fe', 'Zr', 'V', 'Cr', 'Mn', 'Ni', 'Co', 'Y']

df3_std = df3.rename(columns={"Pa": "P_abs", "Pd": "P_des"})

# normalize composition rows to sum = 1
row_sum = df3_std[element_cols3].sum(axis=1)
df3_std[element_cols3] = df3_std[element_cols3].div(row_sum, axis=0)

df3_std["table"] = 3

target_cols = ["table", "wt", "P_abs", "P_des"] + all_elements
df3_std = df3_std.reindex(columns=target_cols, fill_value=0)

# checks
print(df3_std[all_elements].sum(axis=1).describe())   # all ~1.0
df3_std.head()

count    1.590000e+02
mean     1.000000e+00
std      8.698964e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


,table,wt,P_abs,P_des,Ag,Al,Au,B,Be,Bi,...,Tb,Th,Ti,Tm,U,V,W,Y,Zn,Zr
0,3,1.38,0.060,0.040,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.000
1,3,1.42,0.032,0.020,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.025
2,3,1.17,0.027,0.020,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.050
3,3,0.90,0.018,0.018,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.075
4,3,1.70,0.200,0.150,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.000


In [70]:
df3_std["table"] = "3_" + df3["NO"].astype(int).astype(str)
df3_std.head()

,table,wt,P_abs,P_des,Ag,Al,Au,B,Be,Bi,...,Tb,Th,Ti,Tm,U,V,W,Y,Zn,Zr
0,3_1,1.38,0.060,0.040,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.000
1,3_2,1.42,0.032,0.020,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.025
2,3_3,1.17,0.027,0.020,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.050
3,3_4,0.90,0.018,0.018,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.075
4,3_5,1.70,0.200,0.150,0,0,0,0,0,0,...,0,0,0.5,0,0,0.0,0,0.0,0,0.000


In [71]:
df2.columns

Index(['Unnamed: 0', 'index', 'Material_Class', 'Composition_Formula',
       'Hydrogen_Weight_Percent', 'Heat_of_Formation_kJperMolH2',
       'Temperature_oC', 'Pressure_Atmospheres_Absolute',
       'Entropy_of_Formation_JperMolH2perK', 'Equilibrium_Pressure_25C',
       'LnEquilibrium_Pressure_25C', 'HtoM', 'Reference', 'Ag', 'Al', 'Au',
       'B', 'Be', 'Bi', 'C', 'Ca', 'Ce', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'Eu',
       'Fe', 'Ga', 'Gd', 'Ge', 'Hf', 'Ho', 'In', 'Ir', 'La', 'Li', 'Lu', 'Mg',
       'Mn', 'Mo', 'N', 'Nb', 'Nd', 'Ni', 'O', 'Os', 'Pb', 'Pd', 'Pr', 'Pt',
       'Re', 'Rh', 'Ru', 'Sb', 'Sc', 'Si', 'Sm', 'Sn', 'Ta', 'Tb', 'Th', 'Ti',
       'Tm', 'U', 'V', 'W', 'Y', 'Zn', 'Zr'],
      dtype='object')

In [72]:
df2_std = df2.rename(columns={
    "Hydrogen_Weight_Percent": "wt",
    "Pressure_Atmospheres_Absolute": "P_abs",
})
df2_std["table"] = "2_" + (df2_std.index + 1).astype(str)

df2_std = df2_std.reindex(columns=target_cols)          # missing cols -> NaN
df2_std[all_elements] = df2_std[all_elements].fillna(0)  # elements -> 0

merged = pd.concat([df3_std, df2_std], ignore_index=True)
print(merged.shape)
merged.sample(5)

(931, 62)


,table,wt,P_abs,P_des,Ag,Al,Au,B,Be,Bi,...,Tb,Th,Ti,Tm,U,V,W,Y,Zn,Zr
658,2_500,1.523236,NaN,NaN,0.0,0.053333,0,0.0,0.0,0.0,...,0.0,0.0,0.066667,0.0,0.0,0.0,0.0,0.0,0.0,0.266667
900,2_742,1.929227,NaN,NaN,0.0,0.000000,0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.333333
357,2_199,1.380000,0.64,NaN,0.0,0.066667,0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
51,3_52,1.900000,0.20,0.12,0.0,0.000000,0,0.0,0.0,0.0,...,0.0,0.0,0.526316,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
606,2_448,1.836981,NaN,NaN,0.0,0.000000,0,0.0,0.0,0.0,...,0.0,0.0,0.337838,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [73]:
print(merged.shape)

(931, 62)


In [74]:
print(df.columns.tolist())

['Composition', 'hydrogen_weight_percent', 'heat_of_formation', 'temperature', 'pressure', 'avg_atomic_mass', 'Ti', 'V', 'Nb', 'Cr', 'Co', 'Ni', 'Fe', 'Al', 'Mn', 'Zr', 'Cu', 'Ag', 'Pd', 'W', 'Mg', 'Zn', 'Bi', 'Hf', 'Li', 'B', 'Mo', 'Si', 'Rh', 'Pt', 'Ir', 'Y', 'Ce', 'Pb', 'Sb', 'Sc', 'U', 'Ru', 'Ta', 'N', 'C', 'Au', 'Ga', 'In', 'Os', 'Re', 'Sn', 'Ge', 'La']


In [75]:
df_std = df.rename(columns={
    "hydrogen_weight_percent": "wt",
    "pressure": "P_abs",        # assuming this is absorption/equilibrium pressure
})
df_std["table"] = "1_" + (df_std.index + 1).astype(str)

df_std = df_std.reindex(columns=target_cols)           # missing cols -> NaN
df_std[all_elements] = df_std[all_elements].fillna(0)  # elements -> 0

merged = pd.concat([merged, df_std], ignore_index=True)

# checks
print(df_std[all_elements].sum(axis=1).describe())   # ~1.0
print(merged.shape)                                  

count    5.370000e+02
mean     1.000000e+00
std      4.080695e-07
min      9.999980e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000002e+00
dtype: float64
(1468, 62)


In [ ]:
tife = merged[(merged["Ti"] > 0) & (merged["Fe"] > 0)]
print(tife.shape)

(404, 62)


In [77]:
tife_strict = merged[
    (merged["Ti"] + merged["Fe"] >= 0.6) &      # Ti+Fe dominate
    (merged["Ti"].between(0.35, 0.65))           # roughly AB stoichiometry
]
print(tife_strict.shape)

(179, 62)


In [79]:
tife_strict.to_csv("main.csv", index=False)
tife.to_csv("side.csv", index=False)